In [ ]:
!pip install -q diffusers transformers accelerate peft datasets bitsandbytes safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00


In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset
from diffusers import UNet2DConditionModel, DDPMScheduler
from peft import LoraConfig, get_peft_model
import torch.optim as optim

device = "cuda"

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class ImageTextDataset(Dataset):
    def __init__(self, image_dir, text_dir, size=512):
        self.image_dir = image_dir
        self.text_dir = text_dir

        self.images = sorted(os.listdir(image_dir))

        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_name = self.images[idx]

        # 🔹 Load image
        image_path = os.path.join(self.image_dir, image_name)
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)

        # 🔹 Load matching text
        text_name = os.path.splitext(image_name)[0] + ".txt"
        text_path = os.path.join(self.text_dir, text_name)

        with open(text_path, "r") as f:
            caption = f.read().strip()

        return {
            "pixel_values": image,
            "text": caption
        }

In [ ]:
from torch.utils.data import DataLoader

dataset = ImageTextDataset(
    image_dir="/content/drive/MyDrive/dataset/images",
    text_dir="/content/drive/MyDrive/dataset/captions"
)

dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

In [ ]:
from torchvision import transforms

In [ ]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"

unet = UNet2DConditionModel.from_pretrained(
    model_id,
    subfolder="unet",
    torch_dtype=torch.float16
).to(device)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"]
)

unet = get_peft_model(unet, lora_config)

In [ ]:
noise_scheduler = DDPMScheduler.from_pretrained(
    model_id,
    subfolder="scheduler"
)

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

In [ ]:
optimizer = optim.Adam(unet.parameters(), lr=1e-4)

In [ ]:
from transformers import CLIPTokenizer, CLIPTextModel

tokenizer = CLIPTokenizer.from_pretrained(
    "openai/clip-vit-large-patch14"
)

text_encoder = CLIPTextModel.from_pretrained(
    "openai/clip-vit-large-patch14"
).to(device)

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}

In [ ]:
encoder_hidden_states = text_encoder(
    inputs.input_ids
).last_hidden_state

encoder_hidden_states = encoder_hidden_states.to(unet.dtype)

In [ ]:
images = images.to(device, dtype=unet.dtype)

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from diffusers import (
    UNet2DConditionModel,
    AutoencoderKL,
    DDPMScheduler
)
from transformers import CLIPTokenizer, CLIPTextModel
from torchvision import transforms
from PIL import Image
import os

# =====================================================
# DEVICE
# =====================================================
device = "cuda"

# =====================================================
# LOAD MODELS (FP16)
# =====================================================

model_id = "runwayml/stable-diffusion-v1-5"

tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")

text_encoder = CLIPTextModel.from_pretrained(
    model_id,
    subfolder="text_encoder",
    torch_dtype=torch.float16
).to(device)

vae = AutoencoderKL.from_pretrained(
    model_id,
    subfolder="vae",
    torch_dtype=torch.float16
).to(device)

unet = UNet2DConditionModel.from_pretrained(
    model_id,
    subfolder="unet",
    torch_dtype=torch.float16
).to(device)

noise_scheduler = DDPMScheduler.from_pretrained(
    model_id,
    subfolder="scheduler"
)

# Freeze VAE and text encoder (recommended)
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
text_encoder.eval()

# =====================================================
# DATASET
# =====================================================

class ImageTextDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, text_dir, size=512):
        self.image_dir = image_dir
        self.text_dir = text_dir
        self.images = sorted(os.listdir(image_dir))

        self.transform = transforms.Compose([
           transforms.Resize((size, size)),
           transforms.Grayscale(num_output_channels=3),
           transforms.ToTensor(),
           transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5])
           ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_name = self.images[idx]

        image = Image.open(
            os.path.join(self.image_dir, image_name)
        ).convert("RGB")

        image = self.transform(image)

        text_name = os.path.splitext(image_name)[0] + ".txt"
        with open(os.path.join(self.text_dir, text_name), "r") as f:
            caption = f.read().strip()

        return {
            "pixel_values": image,
            "text": caption
        }

# =====================================================
# DATALOADER
# =====================================================

dataset = ImageTextDataset(
    image_dir="/content/drive/MyDrive/dataset/images",
    text_dir="/content/drive/MyDrive/dataset/captions"
)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# =====================================================
# OPTIMIZER
# =====================================================

optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-5)

# =====================================================
# TRAINING LOOP (FP16)
# =====================================================

epochs = 1

for epoch in range(epochs):

    for step, batch in enumerate(dataloader):

        images = batch["pixel_values"].to(
        device=device,
        dtype=torch.float16  # 🔥 FIX HERE
        )

        # -------------------------------------------------
        # IMAGE → LATENTS (VAE)
        # -------------------------------------------------


        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample()

        latents = latents * 0.18215
        latents = latents.to(dtype=torch.float16)

        # -------------------------------------------------
        # TEXT → EMBEDDINGS
        # -------------------------------------------------
        captions = batch["text"]

        inputs = tokenizer(
            captions,
            padding="max_length",
            truncation=True,
            max_length=77,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            encoder_hidden_states = text_encoder(
                inputs.input_ids
            ).last_hidden_state

        encoder_hidden_states = encoder_hidden_states.to(
            dtype=torch.float16
        )

        # -------------------------------------------------
        # DIFFUSION PROCESS
        # -------------------------------------------------
        noise = torch.randn_like(latents)

        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (latents.shape[0],),
            device=device
        ).long()

        noisy_latents = noise_scheduler.add_noise(
            latents,
            noise,
            timesteps
        )

        # -------------------------------------------------
        # UNET PREDICTION
        # -------------------------------------------------
        noise_pred = unet(
            noisy_latents,
            timesteps,
            encoder_hidden_states=encoder_hidden_states
        ).sample

        loss = F.mse_loss(noise_pred, noise)

        # -------------------------------------------------
        # BACKPROP
        # -------------------------------------------------
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 10 == 0:
            print(f"Epoch {epoch} Step {step} Loss {loss.item():.4f}")

print("Training complete!")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: runwayml/stable-diffusion-v1-5
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Epoch 0 Step 0 Loss 0.1329
Epoch 0 Step 10 Loss nan
Epoch 0 Step 20 Loss nan
Epoch 0 Step 30 Loss nan
Epoch 0 Step 40 Loss nan
Epoch 0 Step 50 Loss nan
Epoch 0 Step 60 Loss nan
Epoch 0 Step 70 Loss nan
Epoch 0 Step 80 Loss nan
Epoch 0 Step 90 Loss nan
Epoch 0 Step 100 Loss nan
Epoch 0 Step 110 Loss nan
Epoch 0 Step 120 Loss nan
Epoch 0 Step 130 Loss nan
Epoch 0 Step 140 Loss nan
Epoch 0 Step 150 Loss nan
Epoch 0 Step 160 Loss nan
Epoch 0 Step 170 Loss nan
Epoch 0 Step 180 Loss nan
Epoch 0 Step 190 Loss nan
Epoch 0 Step 200 Loss nan
Epoch 0 Step 210 Loss nan
Epoch 0 Step 220 Loss nan
Epoch 0 Step 230 Loss nan
Epoch 0 Step 240 Loss nan
Epoch 0 Step 250 Loss nan
Epoch 0 Step 260 Loss nan
Epoch 0 Step 270 Loss nan
Epoch 0 Step 280 Loss nan
Epoch 0 Step 290 Loss nan
Epoch 0 Step 300 Loss nan
Epoch 0 Step 310 Loss nan
Epoch 0 Step 320 Loss nan
Epoch 0 Step 330 Loss nan
Epoch 0 Step 340 Loss nan
Epoch 0 Step 350 Loss nan
Epoch 0 Step 360 Loss nan
Epoch 0 Step 370 Loss nan
Epoch 0 Step 380 Los

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to("cuda")

image = pipe("Sagittal T1-weighted brain MRI scan of an Alzheimer's disease patient grayscale").images[0]
image.save("generated.png")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/50 [00:00<?, ?it/s]